# Notebook 03 — Customer Metrics & Retention Base

Objective:
Build customer-level behavioral metrics to enable:
- Retention analysis
- Cohort analysis
- Purchase frequency analysis
- Customer lifetime value (LTV)
- Churn proxy definition

Dataset:
Cleaned and filtered to delivered orders only.

### Imports

In [1]:
import pandas as pd
import numpy as np

### Loaded Cleaned Dataset

In [2]:
df = pd.read_csv("../data/processed/fact_orders.csv")

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,order_value,review_score,delivery_delay_days,order_year,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,38.71,4.0,-8.0,2017,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,141.46,4.0,-6.0,2018,7
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,179.12,5.0,-18.0,2018,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,72.20,5.0,-13.0,2017,11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,28.62,5.0,-10.0,2018,2


### Ensure date format

In [3]:
date_cols = [
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96468 entries, 0 to 96467
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       96468 non-null  object        
 1   customer_id                    96468 non-null  object        
 2   order_status                   96468 non-null  object        
 3   order_purchase_timestamp       96468 non-null  datetime64[ns]
 4   order_approved_at              96468 non-null  object        
 5   order_delivered_carrier_date   96468 non-null  object        
 6   order_delivered_customer_date  96468 non-null  datetime64[ns]
 7   order_estimated_delivery_date  96468 non-null  datetime64[ns]
 8   customer_unique_id             96468 non-null  object        
 9   order_value                    96468 non-null  float64       
 10  review_score                   95822 non-null  float64       
 11  delivery_delay_

In [4]:
# For this task, we will focus on the dates columns order_purchase_timestamp, order_delivered_customer_date, order_estimated_delivery_date.
# So order_approved_at, order_delivered_carrier_date will be dropped.
df = df.drop(columns=["order_approved_at", "order_delivered_carrier_date"])

### Customer Aggregation

In [5]:
# Customer Level Aggregation
customer_metrics = df.groupby("customer_unique_id").agg(
    total_orders=("order_id", "nunique"),
    total_revenue=("order_value", "sum"),
    avg_order_value=("order_value", "mean"),
    first_purchase =("order_purchase_timestamp", "min"),
    last_purchase =("order_purchase_timestamp", "max"),
    avg_review_score = ("review_score", "mean")
).reset_index()

customer_metrics.head()

,customer_unique_id,total_orders,total_revenue,avg_order_value,first_purchase,last_purchase,avg_review_score
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,141.90,2018-05-10 10:56:27,2018-05-10 10:56:27,5.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,27.19,2018-05-07 11:11:27,2018-05-07 11:11:27,4.0
2,0000f46a3911fa3c0805444483337064,1,86.22,86.22,2017-03-10 21:05:03,2017-03-10 21:05:03,3.0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,43.62,2017-10-12 20:29:41,2017-10-12 20:29:41,4.0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,196.89,2017-11-14 19:45:42,2017-11-14 19:45:42,5.0


In [6]:
# Add Customer Lifetime
customer_metrics["customer_lifetime_days"] = (customer_metrics["last_purchase"] - customer_metrics["first_purchase"]).dt.days

In [7]:
# Purchase Frequency
customer_metrics["purchase_frequency"] = customer_metrics["total_orders"] / (customer_metrics["customer_lifetime_days"] + 1)

In [ ]:
# Calcular días desde la última compra (Recency)
snapshot_date = df["order_purchase_timestamp"].max()

customer_metrics["days_since_last_purchase"] = (snapshot_date - customer_metrics["last_purchase"]).dt.days

### Churn Proxy

In [ ]:
# Define Churn Threshold (e.g., 90 days without purchase)
churn_threshold = 90
customer_metrics["churned"] = np.where(customer_metrics["days_since_last_purchase"] > churn_threshold, 1, 0)

In [10]:
# Churn Rate
churn_rate = customer_metrics["churned"].mean()
print(f"Customer Churn Rate: {churn_rate:.2%}")

Customer Churn Rate: 80.09%


### Cohort Base

In [11]:
# Cohort Month
customer_metrics["cohort_month"] = customer_metrics["first_purchase"].dt.to_period("M")

In [12]:
# Order Month
df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")

In [13]:
# Merge Cohort info
df = df.merge(customer_metrics[["customer_unique_id", "cohort_month"]], on="customer_unique_id", how="left")

In [14]:
# Cohort index
df["cohort_index"] = (df["order_month"] - df["cohort_month"]).apply(lambda x: x.n)

### Retention Table

In [16]:
# Cohort Retention Base
cohort_data = df.groupby(["cohort_month", "cohort_index"]).agg(
    active_customers=("customer_unique_id", "nunique")
).reset_index()

cohort_sizes = cohort_data[cohort_data["cohort_index"] == 0][
    ["cohort_month", "active_customers"]].rename(columns={"active_customers": "cohort_size"})

cohort_data = cohort_data.merge(cohort_sizes, on="cohort_month")

cohort_data["retention_rate"] = (
    cohort_data["active_customers"] / cohort_data["cohort_size"]
)

cohort_data.head()

,cohort_month,cohort_index,active_customers,cohort_size,retention_rate
0,2016-10,0,262,262,1.000000
1,2016-10,6,1,262,0.003817
2,2016-10,9,1,262,0.003817
3,2016-10,11,1,262,0.003817
4,2016-10,13,1,262,0.003817


In [19]:
# Final export of customer metrics and cohort data
customer_metrics.to_csv("../data/processed/customer_metrics.csv", index=False)

cohort_data.to_csv("../data/processed/cohort_retention.csv", index=False)

## Customer Metrics Summary

This notebook created a customer-level behavioral dataset including:

- Total orders
- Total revenue
- Average order value
- Customer lifetime
- Purchase frequency
- Recency
- Churn proxy flag
- Cohort assignment

This dataset enables:

- Retention analysis
- Cohort modeling
- LTV estimation
- Future predictive modeling
- SQL warehouse modeling
- Power BI customer dashboards

Project moving from data preparation to behavioral intelligence.

### 